# MLP Analisys

In [ ]:
# === STEP 1: Import e configurazione Spark ===

# Standard library
import time
import json
from itertools import product

# Spark core
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, StructField

# Spark MLlib — feature engineering
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Spark MLlib — modello
from pyspark.ml.classification import MultilayerPerceptronClassifier, MultilayerPerceptronClassificationModel
# Spark MLlib — valutazione
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Spark MLlib — pipeline
from pyspark.ml import Pipeline
from pyspark.ml import PipelineModel

# Pucktrick
from pucktrick import PuckTrick, Engine

# Solo per export finale — NON usato nel loop sperimentale
import pandas as pd

## Step 2 — Configurazione Spark e caricamento dataset

In questa sezione inizializziamo la `SparkSession` connettendoci al cluster remoto e carichiamo il dataset **MetroPT-3**.

Il preprocessing si limita a:
- **Selezione** delle sole colonne necessarie agli esperimenti: le 3 feature ad alta importanza identificate nell'analisi esplorativa (`DV_pressure`, `Oil_temperature`, `TP3`) e la label `target`
- **Cast** di tutte le colonne a `DoubleType`, requisito di Spark MLlib per feature e label
- **Drop dei null** per garantire consistenza tra i run

Infine, inizializziamo `PuckTrick` con backend Spark: questo aggiunge automaticamente la colonna `_pucktrick_row_id` al DataFrame, necessaria per tracciare le righe modificate durante l'iniezione del rumore. Il DataFrame risultante (`base_df`) è la **base immutabile** da cui partono tutti gli esperimenti — non verrà mai modificato direttamente.

### Costanti sperimentali

| Parametro | Valore |
|---|---|
| Feature selezionate | `DV_pressure_scaled`, `Oil_temperature_scaled`, `TP3_scaled` |
| Tipi di rumore | `duplicated`, `labels`, `missing`, `noisy`, `outliers` |
| Livelli di rumore | 0%, 10%, 20%, 30%, 50% |
| Run per combinazione | 20 |
| t di Student (95%, df=19) | 2.093 |

### Nota statistica — Intervallo di Confidenza al 95%

Per ogni combinazione *(tipo di rumore × feature × percentuale)* eseguiamo **20 run indipendenti**,
ciascuno con un seed diverso. Questo ci permette di stimare non solo la metrica media,
ma anche la sua **variabilità** tramite l'intervallo di confidenza al 95%.

La formula utilizzata è quella dell'intervallo di confidenza per campioni piccoli (distribuzione t di Student):

$$IC = \bar{x} \pm t \cdot \frac{s}{\sqrt{n}}$$

Dove:
- $\bar{x}$ è la **media** dell'F1-score sui 20 run
- $s$ è la **deviazione standard** campionaria
- $n = 20$ è il numero di run
- $t = 2.093$ è il valore critico dalla **tavola t di Student** con $df = n - 1 = 19$ gradi di libertà e livello di confidenza al 95%

**Esempio pratico:** se sui 20 run otteniamo F1 medio $= 0.85$ con deviazione standard $= 0.03$:

$$IC = 0.85 \pm 2.093 \cdot \frac{0.03}{\sqrt{20}} = 0.85 \pm 0.014 = [0.836,\ 0.864]$$

Questo significa che siamo al 95% confidenti che il vero valore dell'F1 — quello che otterremmo
con infiniti run — cada nell'intervallo $[0.836, 0.864]$.

Confrontando gli intervalli tra livelli di rumore diversi, possiamo stabilire se una degradazione
delle performance è **statisticamente significativa** o rientra nella variabilità naturale del modello.

> **Nota:** il valore $t = 2.093$ è valido **solo con $n = 20$ run**. Se si modifica `N_RUNS`,
> aggiornare `T_VALUE_95` consultando la tavola t di Student alla riga $df = N\_RUNS - 1$.

Link alla tavola student
https://www.statisticshowto.com/tables/t-distribution-table/

In [ ]:
# === STEP 2: Configurazione Spark e caricamento dataset ===

# ── Configurazione cluster ─────────────────────────────────────────────────
MASTER_URL  = "spark://10.0.1.8:7077"
DRIVER_HOST = "10.0.1.8"

spark = SparkSession.builder \
    .appName("MetroPT_MLP_Experiments") \
    .master(MASTER_URL) \
    .config("spark.submit.deployMode",      "client") \
    .config("spark.executor.instances",     "4") \
    .config("spark.executor.cores",         "4") \
    .config("spark.executor.memory",        "13g") \
    .config("spark.driver.memory",          "8g") \
    .config("spark.driver.host",            DRIVER_HOST) \
    .config("spark.driver.bindAddress",     DRIVER_HOST) \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession creata — versione:", spark.version)

# ── Costanti sperimentali ──────────────────────────────────────────────────
ALL_FEATURES = [
    "TP2_scaled", "TP3_scaled", "H1_scaled", "DV_pressure_scaled",
    "Reservoirs_scaled", "Oil_temperature_scaled", "Motor_current_scaled",
    "COMP", "DV_eletric", "Towers", "MPG", "LPS", "Pressure_switch"
]
NOISE_FEATURES = ["DV_pressure_scaled", "Oil_temperature_scaled", "TP3_scaled"]
LABEL_COL    = "target"
NOISE_TYPES  = ["duplicated", "labels", "missing", "noisy", "outliers"]
NOISE_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.5]
N_RUNS       = 20
T_VALUE_95   = 2.093

# ── Caricamento dataset ────────────────────────────────────────────────────
# Nota: il file è stato salvato con pandas to_parquet (file singolo, non directory Spark).
# Si legge sul driver e si distribuisce immediatamente sul cluster via createDataFrame.
DATA_PATH = "/home/PuckTrickadmin/DATASETS/MetroDT_Modified.parquet"
raw_pdf = pd.read_parquet(DATA_PATH)
raw_df = spark.createDataFrame(raw_pdf)
del raw_pdf  # libera subito memoria driver

# ── Selezione colonne e cast ───────────────────────────────────────────────
df = raw_df.select(
    F.col("timestamp"),
    F.col("TP2_scaled").cast(DoubleType()),
    F.col("TP3_scaled").cast(DoubleType()),
    F.col("H1_scaled").cast(DoubleType()),
    F.col("DV_pressure_scaled").cast(DoubleType()),
    F.col("Reservoirs_scaled").cast(DoubleType()),
    F.col("Oil_temperature_scaled").cast(DoubleType()),
    F.col("Motor_current_scaled").cast(DoubleType()),
    F.col("COMP").cast(DoubleType()),
    F.col("DV_eletric").cast(DoubleType()),
    F.col("Towers").cast(DoubleType()),
    F.col("MPG").cast(DoubleType()),
    F.col("LPS").cast(DoubleType()),
    F.col("Pressure_switch").cast(DoubleType()),
    F.col("target").cast(DoubleType())
).dropna()
# ── Split temporale — coerente con il preprocessing ───────────────────────
SPLIT_DATE = "2020-06-01 00:00:00"

pt         = PuckTrick(df, engine=Engine.SPARK)
base_df    = pt.original

base_train_df = base_df.filter(F.col("timestamp") < SPLIT_DATE).drop("timestamp")
base_test_df  = base_df.filter(F.col("timestamp") >= SPLIT_DATE).drop("timestamp")

base_train_df.cache()
base_test_df.cache()

n_train       = base_train_df.count()
n_test        = base_test_df.count()
n_train_fault = base_train_df.filter(F.col("target") == 1).count()
n_test_fault  = base_test_df.filter(F.col("target") == 1).count()

print(f"Training : {n_train:,} righe ({n_train_fault:,} guasti, {n_train_fault/n_train*100:.2f}%)")
print(f"Test     : {n_test:,} righe ({n_test_fault:,} guasti, {n_test_fault/n_test*100:.2f}%)")
print(f"Split    : {n_train/(n_train+n_test)*100:.1f}% train / {n_test/(n_train+n_test)*100:.1f}% test")
print(f"\nAtteso dal preprocessing:")
print(f"  Training: ~856,832 righe, ~1.29% guasti")
print(f"  Test    : ~660,116 righe, ~2.87% guasti")

## Step 3 — Tuning iperparametri MLP sul dato pulito

### Perché il tuning è necessario

Il `MultilayerPerceptronClassifier` di Spark MLlib espone diversi iperparametri
che influenzano significativamente le performance del modello. Sceglierli in modo
arbitrario rischierebbe di introdurre un bias nei risultati: un modello mal
configurato potrebbe degradare non per il rumore introdotto, ma semplicemente
perché l'architettura è inadatta al problema.

Per questo motivo eseguiamo una sessione di tuning **una volta sola, sul dato
pulito**, e fissiamo gli iperparametri trovati per tutti i 3.000 run sperimentali.
Questo garantisce che qualsiasi variazione di F1-score osservata nei capitoli
successivi sia attribuibile **esclusivamente al rumore** e non a differenze di
configurazione del modello.

### Architettura del MLP

Il `MultilayerPerceptronClassifier` di Spark MLlib implementa un percettrone
multistrato con funzione di attivazione **sigmoide** per gli strati nascosti e
**softmax** per lo strato di output. Il parametro `layers` definisce la struttura
completa della rete come lista di interi, dove ogni elemento rappresenta il numero
di neuroni in quello strato.

Nel nostro caso:
- **Input layer**: dimensione fissa **3**, una per ciascuna feature
  (`DV_pressure_scaled`, `Oil_temperature_scaled`, `TP3_scaled`)
- **Hidden layer(s)**: da ottimizzare
- **Output layer**: dimensione fissa **2**, una per ciascuna classe
  (classe 0 = normale, classe 1 = guasto)

### Iperparametri da ottimizzare

| Iperparametro | Descrizione | Valori testati |
|---|---|---|
| `layers` | Architettura completa della rete | `[13,16,2]`, `[13,32,2]`, `[13,64,2]`, `[13,32,16,2]`, `[13,64,32,16,2]` |
| `maxIter` | Numero massimo di iterazioni di ottimizzazione | `50`, `100` |
| `stepSize` | Learning rate dell'ottimizzatore L-BFGS | `0.01`, `0.05` |
| `blockSize` | Dimensione del blocco per aggregazione gradienti | `128` (fisso) |

> **Nota su `blockSize`:** controlla quanti campioni vengono aggregati insieme
> prima di aggiornare i pesi. Valori più alti migliorano l'efficienza su cluster
> ma richiedono più memoria per executor. `128` è il default consigliato da Spark
> per dataset di questa dimensione.

> **Nota su `stepSize`:** a differenza del learning rate nel SGD classico, qui
> controlla il passo iniziale dell'ottimizzatore **L-BFGS** usato internamente
> da Spark MLlib. L-BFGS è un metodo quasi-Newton che converge più velocemente
> del gradient descent per problemi con poche feature — adatto al nostro caso
> con sole 3 feature in input.

### Strategia di ricerca

La ricerca avviene tramite **`CrossValidator`** con **3 fold** applicato su un
campione del **20% del training set**. Ridurre il dataset per il tuning è una
scelta deliberata: con 856.832 righe, eseguire 16 combinazioni × 3 fold sul
dataset completo richiederebbe tempi eccessivi senza un reale beneficio,
dato che il modello converge su campioni molto più piccoli.

Il test set è **completamente escluso** dal tuning — viene usato solo alla fine
di questo step per calcolare la **baseline F1** sul dato pulito a 0% di rumore,
che costituisce il riferimento per tutti i risultati del Capitolo 4.

La metrica di selezione è **F1-score weighted**: tiene conto dello sbilanciamento
tra classi (98% normale, 2% guasto) pesando il contributo di ciascuna classe
proporzionalmente alla sua frequenza, senza ignorare la classe minoritaria come
farebbe l'accuracy.

### Output atteso

Al termine di questo step otteniamo:
- Le costanti `BEST_LAYERS`, `BEST_MAX_ITER`, `BEST_STEP` — fissate per tutti i run successivi
- La **baseline F1** sul test set pulito — il punto di riferimento del Capitolo 4

In [ ]:
# === STEP 3: Tuning iperparametri MLP sul dato pulito ===
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ── Campione per il tuning (20% del training set) ─────────────────────────
tune_df, _ = base_train_df.randomSplit([0.2, 0.8], seed=42)
tune_df.cache()
print(f"Righe per tuning: {tune_df.count():,}")

# ── VectorAssembler ────────────────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=ALL_FEATURES,
    outputCol="features"
)

# ── Modello base ───────────────────────────────────────────────────────────
mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    blockSize=128,
    seed=42
)

# ── Pipeline ───────────────────────────────────────────────────────────────
pipeline = Pipeline(stages=[assembler, mlp])

# ── Griglia iperparametri ──────────────────────────────────────────────────
param_grid = ParamGridBuilder() \
    .addGrid(mlp.layers, [
        [13, 32, 2],
        [13, 64, 2],
        [13, 32, 16, 2],
        [13, 64, 32, 16, 2]   # architettura proposta
    ]) \
    .addGrid(mlp.maxIter, [50, 100]) \
    .addGrid(mlp.stepSize, [0.01, 0.05]) \
    .build()

print(f"Combinazioni da testare: {len(param_grid)}")

# ── Evaluator ──────────────────────────────────────────────────────────────
evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="f1"
)

# ── CrossValidator ─────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=4
)

# ── Esecuzione tuning ──────────────────────────────────────────────────────
print("Avvio tuning...")
t0 = time.time()
cv_model = cv.fit(tune_df)
print(f"Tuning completato in {(time.time()-t0)/60:.1f} minuti")

# ── Estrazione migliori iperparametri ──────────────────────────────────────
best_pipeline_model: PipelineModel = cv_model.bestModel  # type: ignore
best_mlp: MultilayerPerceptronClassificationModel = best_pipeline_model.stages[-1]  # type: ignore

BEST_LAYERS   = best_mlp.getLayers()
BEST_MAX_ITER = best_mlp.getMaxIter()
BEST_STEP     = best_mlp.getStepSize()

print("\n=== MIGLIORI IPERPARAMETRI ===")
print(f"  layers   : {BEST_LAYERS}")
print(f"  maxIter  : {BEST_MAX_ITER}")
print(f"  stepSize : {BEST_STEP}")
print(f"  blockSize: 128 (fisso)")

# ── Baseline F1 sul test set pulito ───────────────────────────────────────
baseline_pred = best_pipeline_model.transform(base_test_df)
baseline_f1   = evaluator.evaluate(baseline_pred)

print(f"\n=== BASELINE (0% rumore) ===")
print(f"  F1-score sul test set pulito: {baseline_f1:.4f}")

# === SALVATAGGIO IPERPARAMETRI TUNING ===
import json

PARAMS_PATH = "/home/PuckTrickadmin/Daniel/PARAMS/mlp_best_params.json"

model_params = {
    "layers":      list(BEST_LAYERS),
    "maxIter":     BEST_MAX_ITER,
    "stepSize":    BEST_STEP,
    "blockSize":   128,
    "baseline_f1": baseline_f1
}

with open(PARAMS_PATH, "w") as f:
    json.dump(model_params, f, indent=2)

print(f"✅ Parametri salvati in: {PARAMS_PATH}")
print(json.dumps(model_params, indent=2))

---

In [ ]:
# # === CARICAMENTO IPERPARAMETRI TUNING ===
# import json

# PARAMS_PATH = "/home/PuckTrickadmin/DATASETS/mlp_best_params.json"

# with open(PARAMS_PATH, "r") as f:
#     model_params = json.load(f)

# BEST_LAYERS   = model_params["layers"]
# BEST_MAX_ITER = model_params["maxIter"]
# BEST_STEP     = model_params["stepSize"]

# print(f"✅ Parametri caricati:")
# print(f"  layers   : {BEST_LAYERS}")
# print(f"  maxIter  : {BEST_MAX_ITER}")
# print(f"  stepSize : {BEST_STEP}")